# AgentIR: preserving historical evidence in the final 10 images

**One final focused diagnostic before the meeting.** Reuses the previous candidate-memory results. No Qwen calls or API key needed. Development runs use cached text and image embeddings without loading BLIP again.

The previous experiment recovered targets into the pool but often lost them when selecting 10 images. Here we keep **exactly the same 50-image pools** and change **only the selection rule**.

Two pools: **original + current RRF**, and **all-history RRF**. Two saved stages: after 4 and 9 answers. The primary stage is after 9 answers. The same 100 targets are development data, not a fresh confirmation.

| Selector | How it picks exactly 10 images |
|---|---|
| Current MMR | Existing baseline: current-query relevance plus image-vector diversity |
| History MMR | Same algorithm and weight, but relevance comes from the fused-history ranking |
| History top 10 | Take the first 10 images in the fused ranking, without a diversity term |
| Split 5 + 5 | Five highest-current-ranked images in the pool, then five unseen images ranked by earlier history |

MMR uses weight 0.7 and the same rank-to-relevance transformation in both variants. For the split rule, earlier history means original only for the original + current pool, or all earlier unique descriptions for the all-history pool. Every selection stays inside the saved pool. No target labels enter selection.

This is a controlled selector experiment, **not an agent or question-generation experiment**, and MMR is neither PlugIR clustering nor A2R DPP. Target inclusion is a diagnostic, not proof of useful questions.

Files: this notebook and `evidence_selection_tools.zip`. The prior data ZIP and candidate-memory run should already be in Drive. If not, the notebook prompts for the prior data ZIP and `AgentIR_candidate_memory_results.zip`.


## 1. Install packages
Use the same Colab T4 GPU setting as before. Development needs no model download. The optional holdout step uses BLIP to encode fresh saved queries.

In [ ]:
%pip -q install "transformers==4.51.3" "huggingface-hub==0.36.2" "pandas>=2,<4" matplotlib


## 2. Connect Drive and locate the previous run
Change MEMORY_RUN_NAME only if you named the previous run differently. Keep this notebook’s outputs separate.

In [ ]:
import os,json,zipfile,hashlib,sys,subprocess
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HOME']='/content/agentir_model_cache'
from pathlib import Path
from google.colab import drive,files
drive.mount('/content/drive')
import torch
if not torch.cuda.is_available():raise RuntimeError('Select a T4 GPU runtime, as in the previous notebook.')
print('GPU:',torch.cuda.get_device_name(0))
MEMORY_RUN_NAME='memory_v1'
SOURCE_ROOT=Path('/content/drive/MyDrive/AgentIR_candidate_memory/runs')/MEMORY_RUN_NAME
PROJECT_DIR=Path('/content/drive/MyDrive/AgentIR_evidence_selection')
RUN_NAME='selection_v1'
RUN_DIR=PROJECT_DIR/'runs'/RUN_NAME
RUN_DIR.mkdir(parents=True,exist_ok=True)
DATA_DIR=Path('/content/agentir_selection_data');DATA_DIR.mkdir(exist_ok=True)
WORK_DIR=Path('/content/agentir_selection_tools');WORK_DIR.mkdir(exist_ok=True)


## 3. Load the small tools ZIP and existing data
Upload `evidence_selection_tools.zip` when prompted. The notebook reuses the large data ZIP automatically. If the previous run is unavailable, upload its results ZIP when asked.

In [ ]:
def load_zip(path,expected,destination):
    if not path.exists():
        print('Upload:',path.name)
        uploaded=files.upload()
        matches=[v for k,v in uploaded.items() if k==path.name]
        if len(matches)!=1:raise ValueError('Expected '+path.name)
        if hashlib.sha256(matches[0]).hexdigest()!=expected:raise ValueError('ZIP version mismatch')
        path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(matches[0])
    if hashlib.sha256(path.read_bytes()).hexdigest()!=expected:raise ValueError('ZIP version mismatch')
    with zipfile.ZipFile(path) as z:
        if any(Path(n).name!=n for n in z.namelist()):raise ValueError('Unexpected paths')
        z.extractall(destination)
load_zip(Path('/content/drive/MyDrive/AgentIR_multi_query/multi_query_pilot_bundle.zip'),'4418ddc07374ab4b42d1c0b97f034afc3580e9adf28d76e6f81da2087554b9db',DATA_DIR)
load_zip(PROJECT_DIR/'evidence_selection_tools.zip','8a93fc54af86ed0fc3db41419bf2ee063cfc81c635f5b5b2cacdc5be5ff42e4e',WORK_DIR)
if not (SOURCE_ROOT/'development/examples.json').exists():
    print('Upload AgentIR_candidate_memory_results.zip')
    uploaded=files.upload()
    matches=[v for k,v in uploaded.items() if k=='AgentIR_candidate_memory_results.zip']
    if len(matches)!=1:raise ValueError('Expected the previous candidate-memory results ZIP')
    import io
    SOURCE_ROOT=PROJECT_DIR/'imported_memory_results'
    SOURCE_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(matches[0])) as z:
        for n in z.namelist():
            p=Path(n)
            if p.is_absolute() or '..' in p.parts:raise ValueError('Unsafe archive path')
        z.extractall(SOURCE_ROOT)
for split in ['smoke','development']:
    for name in ['config.json','examples.json','text_features.npz']:
        if not (SOURCE_ROOT/split/name).exists():raise FileNotFoundError(SOURCE_ROOT/split/name)
print('Using previous results:',SOURCE_ROOT)


## 4. Smoke check, then development run
The script verifies that the old selection and fused top-10 ordering reproduce exactly. If either differs, it stops rather than silently comparing different baselines. The candidate pools stay byte-for-byte identical to the previous result records.

In [ ]:
def run_logged(command,folder):
    folder.mkdir(parents=True,exist_ok=True)
    with (folder/'run.log').open('a') as log:
        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:print(line,end='');log.write(line);log.flush()
        status=proc.wait()
    if status:raise RuntimeError(f'Run failed: see the error above or {folder / "run.log"}')

def run_selection(split,source=None,frozen=None):
    folder=RUN_DIR/split
    command=[sys.executable,'-u',str(WORK_DIR/'selection_experiment.py'),
             '--data',str(DATA_DIR),'--source',str(source or SOURCE_ROOT/split),
             '--output',str(folder),'--split',split,'--device','cuda']
    if frozen:command.extend(['--frozen',str(frozen)])
    run_logged(command,folder)
    return folder
SMOKE_DIR=run_selection('smoke')
DEV_DIR=run_selection('development')


## 5. Results and figures

**Selected-10 coverage:** percentage of targets present in the 10 evidence images. **Pool recoveries selected:** of the targets recovered into this pool from outside the current top 50, how many now survive selection?

Compare **History MMR against Current MMR** to isolate the relevance signal. Then compare against **History top 10** to see whether diversity adds anything beyond simply taking the fused ranking. Examine losses as well as recoveries. Pool coverage must stay constant across selectors.

Mean pairwise cosine is an embedding-similarity diagnostic. Lower values indicate greater embedding diversity, not necessarily more useful visual differences or better questions. Confidence intervals are paired bootstrap estimates and are not corrected for multiple comparisons.


In [ ]:
import pandas as pd
summary=pd.read_csv(DEV_DIR/'summary.csv')
comparisons=pd.read_csv(DEV_DIR/'comparisons.csv')
display(summary.round(3))
print('After 9 answers: versus the previous selector')
display(comparisons.query('answers == 9 and baseline == "current_mmr"').round(2))
print('After 9 answers: versus taking the fused top 10 directly')
display(comparisons.query('answers == 9 and baseline == "history_top10"').round(2))


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(13,5),sharey=True)
for ax,pool in zip(axes,['original_current_rrf','all_history_rrf']):
    part=summary.query('pool_method == @pool').pivot(index='selector',columns='answers',values='selected10_coverage_pct')
    part=part.reindex(['current_mmr','history_mmr','history_top10','split_5_5'])
    part.plot.bar(ax=ax,color=['#5D70A6','#126F6B'])
    ax.set_title(pool);ax.set_ylim(0,100);ax.set_ylabel('Targets in selected 10 (%)');ax.set_xlabel('')
    ax.legend(['After 4 answers','After 9 answers'])
plt.tight_layout();fig.savefig(DEV_DIR/'selection_coverage.png',dpi=160,bbox_inches='tight');plt.show()


## 6. Inspect recoveries and losses without downloading images
This cell shows actual selected-image IDs and their current/fused ranks. Image-server failures cannot affect it. These tables support diagnosis but do not replace visual inspection of question usefulness.

In [ ]:
detail=pd.read_csv(DEV_DIR/'per_target.csv')
base=detail.query('selector == "current_mmr"')[['source_index','answers','pool_method','target_in_selected10']]
paired=detail.merge(base,on=['source_index','answers','pool_method'],suffixes=('','_baseline'))
paired['change']=paired.target_in_selected10-paired.target_in_selected10_baseline
cases=paired.query('answers == 9 and change != 0')
display(cases[['source_index','pool_method','selector','change','recovered_and_selected']])
examples=json.loads((DEV_DIR/'examples.json').read_text())
SOURCE_INDEX=int(cases.iloc[0].source_index) if len(cases) else int(detail.iloc[0].source_index)
POOL='original_current_rrf'
for e in examples:
    if e['source_index']==SOURCE_INDEX and e['answers']==9 and e['pool_method']==POOL:
        print(e['selector'],'| Target selected:',bool(e['target_in_selected10']))
        print('Original:',e['original']);print('Current:',e['current'])
        display(pd.DataFrame({'image_id':e['selected_ids'],'current_rank':e['selected_current_ranks'],
                              'fused_rank':e['selected_fused_ranks'],
                              'is_target':[x==e['target_id'] for x in e['selected_ids']]}))


## 7. Optional holdout: only after choosing and freezing a comparison

**Leave this off for now. Download and review development results first.** Pick one pool and one selector based on development, with a stated reason. The holdout evaluates only that pool and the chosen selector alongside Current MMR and History top 10. The main measurement is selected-10 coverage after 9 answers.

The first holdout run reconstructs the fresh targets' pools using the unchanged candidate-memory protocol. It may download BLIP if unavailable. It does not generate text or ask questions. A frozen choice file is saved before this run and cannot be changed under the same RUN_NAME. Do not tune using holdout results; if you do, that sample is no longer a holdout.


In [ ]:
RUN_HOLDOUT=False
CHOSEN_POOL=''       # original_current_rrf or all_history_rrf
CHOSEN_SELECTOR=''   # history_mmr, history_top10 or split_5_5
CHOICE_REASON=''     # brief justification from development results
if RUN_HOLDOUT:
    if CHOSEN_POOL not in ['original_current_rrf','all_history_rrf']:raise ValueError('Choose a pool first')
    if CHOSEN_SELECTOR not in ['history_mmr','history_top10','split_5_5']:raise ValueError('Choose a selector first')
    if not CHOICE_REASON.strip():raise ValueError('Record why this comparison was chosen')
    frozen=RUN_DIR/'frozen_choice.json'
    choice=dict(pool=CHOSEN_POOL,selector=CHOSEN_SELECTOR,reason=CHOICE_REASON,
                primary_stage=9,budget=50,selected_count=10,
                primary_metric='selected10_coverage',
                development_summary_sha256=hashlib.sha256((DEV_DIR/'summary.csv').read_bytes()).hexdigest())
    if frozen.exists() and json.loads(frozen.read_text())!=choice:raise ValueError('A different comparison is already frozen in this run')
    frozen.write_text(json.dumps(choice,indent=2))
    source=RUN_DIR/'holdout_source'
    run_logged([sys.executable,'-u',str(WORK_DIR/'memory_experiment.py'),'--data',str(DATA_DIR),
                '--output',str(source),'--split','holdout','--device','cuda'],source)
    HOLDOUT_DIR=run_selection('holdout',source=source,frozen=frozen)
    display(pd.read_csv(HOLDOUT_DIR/'summary.csv'))
    display(pd.read_csv(HOLDOUT_DIR/'comparisons.csv').query('answers == 9'))
else:
    print('Holdout untouched. Review development results and freeze a choice first.')


## 8. Download results
Send this ZIP back. We can review the development result, choose the final holdout comparison, and then consolidate the meeting slides.

In [ ]:
import shutil
archive=shutil.make_archive('/content/AgentIR_evidence_selection_results','zip',RUN_DIR)
files.download(archive)
